## Read-only rootfs & `no-new-privileges`

Two cheap, high-leverage flags that together blunt most *post-compromise* behaviour — what an attacker can do *after* landing in your container.

**Read-only root filesystem.** Most apps don't need to write to most of their filesystem — they write to a few specific paths (logs, caches, uploads). Marking the rest read-only blocks a whole class of attacks: dropping a webshell, modifying binaries, tampering with config.

```bash
docker run --read-only --tmpfs /tmp --tmpfs /var/run -v applogs:/app/logs myorg/api
```

`--read-only` freezes the rootfs; the `--tmpfs` mounts give back the *specific* writable scratch paths apps expect (`/tmp`, PID files in `/var/run`) as RAM; the volume keeps what you actually want persisted. A few minutes of "what does this app write?" audit buys a much smaller attack surface.

**`no-new-privileges`.** Normally, executing a **setuid** binary can elevate a process. This flag tells the kernel "ignore setuid bits for this process tree — it can never gain privileges beyond what it started with." It breaks `sudo`, `mount`, and other setuid tools inside the container — which for an app container is a *feature*: an attacker who lands there can't setuid their way to root.

```bash
docker run --security-opt no-new-privileges myorg/api
```

Combined with the earlier moves, this is the **hardened baseline** in Compose:

```yaml
services:
  api:
    user: "10001:10001"
    read_only: true
    tmpfs: [/tmp, /var/run]
    cap_drop: [ALL]
    cap_add: [NET_BIND_SERVICE]
    security_opt: [no-new-privileges:true]
```

Non-root, read-only, no capabilities, no escalation — a stack of independent walls, each a one-liner.